[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/connections/connections.ipynb)

# Connections on Curved Surfaces

This example computes connections on curved surfaces: rules for carrying a tangent vector from each point of a surface to its neighbours. The surface's own connection, parallel transport, keeps the vector tangent and turns it no more than the surface itself bends; from it come the holonomy of loops, the connection Laplacian, vector heat diffusion and the smoothest tangent fields. An extra turn on every edge gives the trivial connections of Crane, Desbrun and Schröder, which design tangent vector fields with their singularities exactly where they are wanted, and the advance of a phase along such a field gives the stripe patterns of Knöppel, Crane, Pinkall and Schröder.

The computation is written in the geometric algebra of three-dimensional space. Between two neighbouring points with unit normals `n_tail` and `n_head`, the shortest turn that carries the first normal onto the second is the square root of their geometric product, `(n_head * n_tail).square_root()`, which turns in the plane of the two normals and leaves the hinge line between their tangent planes still. Chained around a closed curve, those rotors turn a tangent vector by the solid angle the normals sweep out; placed into a *sparse extensor* across the edges of a triangle mesh, they form the covariant derivative and the connection Laplacian directly on ambient three-dimensional vectors, with no local two-dimensional coordinate charts or reference edges.

In the notation of differential geometry, carrying a vector along a curve reads as solving $\nabla_{\dot\gamma} X = 0$ with the Levi-Civita connection $\nabla$, the turn around a loop bounding a region $\Omega$ as $\int_\Omega K\,dA$, and the connection Laplacian as $\nabla^*\nabla$.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from itertools import islice

import numpy as np

from numga import concatenate, stack
from numga.algebras import VGA3D as ga
from numga.sparse import SparseExtensor, spdiag
from examples.animation import save_figure, show_animation
from examples.surfaces.connections import render
from examples.surfaces.connections.core import Mesh, as_ga_sparse, as_scalar, faces_around, latitude_rings, loops, mv, spanning_tree
from examples.surfaces.connections.scenarios import face_indices, nearest_faces

Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
duration_ms = 70

## 1. Holonomy around latitude circles

On the unit sphere each point's outward normal is the point itself. A tangent vector carried around a latitude circle, each step turned by the shortest rotor between neighbouring normals, `(n_head * n_tail).square_root()`, stays tangent all the way, and comes back turned by the solid angle of the cap the circle bounds. On the green equator the cap is a hemisphere, a whole turn, and the vector returns as it started; on the blue circle sixty degrees from the pole the cap's solid angle is π, and the vector returns reversed; on the red circle near the pole the cap is small, and the vector turns by about fifty degrees. The faint arrows trail where each vector has been.

In [ ]:
# The sphere's subdivisions, and its three latitude circles, 30, 60 and 90 degrees from the north pole,
# with the steps around each and the steps between frames.
n_subdivisions = 3
colatitudes = np.array([np.pi / 6, np.pi / 3, np.pi / 2])
n_ring_steps = 120
ring_stride = 2

In [ ]:
def transport_step(normal_tail: Vector, normal_head: Vector) -> Rotor:
    """The shortest rotor carrying one unit normal onto the other: transport between their tangent planes."""
    return (normal_head * normal_tail).square_root()                        # [...] Rotor


def transported(normals: Vector, start: Vector) -> Iterator[Vector]:
    """A tangent vector carried along a sequence of unit normals `[steps + 1, ...]`, at each of them."""
    yield start
    for tail, head in zip(normals[:-1], normals[1:]):
        start = transport_step(tail, head) >> start                         # [...] Vector
        yield start


sphere = Mesh.icosphere(n_subdivisions)
rings, starts = latitude_rings(colatitudes, n_ring_steps)                  # [steps + 1, rings] Vector, [rings] Vector
frames = islice(zip(rings, transported(rings, starts)), None, None, ring_stride)
show_animation(render.animate_holonomy(sphere, rings, frames), "connections_holonomy", duration_ms)

## 2. The covariant derivative and connection Laplacian

On a triangle mesh a connection is a rotor on every edge, saying how a tangent vector at the tail is carried into the tangent plane at the head; the surface's own is the shortest turn between the two normals, as on the sphere. The covariant derivative compares, across every edge, the head's vector with the tail's vector carried over: a sparse map from vector fields on the vertices to vectors on the edges, each edge's row holding the rotor's action at its tail and minus one at its head. Weighted by the cotangent ratios of the edges and composed with its adjoint, it gives the connection Laplacian, `covariant.adjoint()(spdiag(weights) * covariant)`, a sparse map between vertex vector fields. Every connection further on, with extra turns on its edges, is built by the same function from its own rotors. Because each rotor carries the tail's normal onto the head's, the Laplacian never mixes tangent and normal components.

In the complex line-bundle notation of discrete differential geometry, each vertex picks an arbitrary reference direction in its tangent plane, each edge rotor reads as a unit complex phase $e^{i\theta_{ij}}$ relating the two reference directions, and the connection Laplacian reads as a Hermitian matrix over complex vertex scalars.

In [ ]:
# The torus: its radii, and the vertices around its hole and around its tube.
major_radius = 2.0
minor_radius = 0.75
n_major = 28
n_minor = 14

In [ ]:
def connection(mesh: Mesh, transports: Rotor, weights: Scalar) -> tuple[SparseExtensor, SparseExtensor]:
    """The covariant derivative of the rotors `[E, 2]` from each end of each edge to its head, and its
    connection Laplacian with each edge weighted by `Scalar[E]`."""
    covariant = as_ga_sparse(mesh.edges, (transports >> Vector) * np.array([-1, 1]))   # [E, V] Vector <- Vector
    laplacian = covariant.adjoint()(spdiag(weights) * covariant)           # [V, V] Vector <- Vector
    return covariant, laplacian


mesh = Mesh.torus(major_radius, minor_radius, n_major, n_minor)
ends = mesh.edges                                                          # [E, 2]
normals = mesh.vertex_normals.batch()                                      # [V] Vector

# From each end of an edge to its head: the dihedral rotor at the tail, and 1 at the head.
turns = transport_step(normals[ends], normals[ends[:, 1, None]])           # [E, 2] Rotor
covariant, laplacian = connection(mesh, turns, mesh.edge_ratio)            # [E, V], [V, V] Vector <- Vector
area_weights = spdiag(mesh.vertex_areas)                                   # [V, V] Scalar

print(covariant.gatype, covariant.shape)
print(laplacian.gatype, laplacian.shape)

## 3. Parallel transport by vector heat diffusion

A tangent bump on the outer equator of the torus diffuses across the surface by backward-Euler heat steps of the connection Laplacian, carried along every path by the connection as it spreads. Normalized at every vertex, the diffused vector is a unit tangent field carried smoothly around the torus; the animation shades the diffusing wave's amplitude.

In [ ]:
# The bump's width, the heat steps' duration, and their number.
source_width = 0.55
step_time = 0.18
n_heat_steps = 20

In [ ]:
def diffusing(source: Vector, n_steps: int) -> Iterator[Vector]:
    """The field after each backward-Euler heat step of the connection Laplacian."""
    step_operator = area_weights * Vector + step_time * laplacian          # [V, V] Vector <- Vector
    for _ in range(n_steps):
        source = step_operator.solve(area_weights * source)                # Vector[V]
        yield source


centre = (mv.x * (major_radius + minor_radius)).dual()                # [] Point
weight = ((mesh.vertices.batch() - centre).norm_squared() * (-0.5 / source_width**2)).exp()   # [V] Scalar
direction = (mv.y + mv.z).normalized()                                     # [] Vector
source = (weight * (direction - normals * (normals | direction))).field()  # Vector[V]
diffused = stack(tuple(diffusing(source, n_heat_steps)))                   # [steps] Vector[V]
transported_field = diffused[-1] / diffused[-1].norm()                     # Vector[V]

show_animation(render.animate_diffusion(mesh, diffused), "connections_diffusion", duration_ms)

## 4. Harmonic tangent vector fields

Because a constant normal field also has zero covariant derivative, a stiff penalty along the normals, `spdiag((unit_normals * (unit_normals | Vector)).field()) * normal_stiffness`, lifts the normal bundle out of the low spectrum without touching tangent vectors. The generalized eigenproblem against the vertex areas, `(laplacian + normal_penalty).eigh(area_weights * Vector, count)`, then finds the smoothest global tangent fields of any connection Laplacian. Turning every tangent vector a quarter turn in its own tangent plane, `u | normals.dual()`, commutes with every edge rotor, so each eigenvalue comes in an equal pair. The figure shows the unit heat-transported field, the fundamental harmonic field and a higher mode.

In [ ]:
# The weight keeping fields out of the normal bundle, and the modes found.
normal_stiffness = 100.0
n_modes = 4

In [ ]:
def low_modes(laplacian: SparseExtensor, normals: Vector, areas: Scalar, normal_stiffness: float, count: int
              ) -> tuple[Scalar, Vector]:
    """The `count` smoothest vector fields of a connection Laplacian against the vertex areas `Scalar[V]`, and
    their energies, with the normal bundle along the unit normals `Vector[V]` lifted out of the low spectrum."""
    unit_normals = normals.batch()                                          # [V] Vector
    normal_penalty = spdiag((unit_normals * (unit_normals | Vector)).field()) * normal_stiffness   # [V, V] Vector <- Vector
    return (laplacian + normal_penalty).eigh(spdiag(areas) * Vector, count)   # [count] Scalar, [count] Vector[V]


mode_energies, modes = low_modes(laplacian, mesh.vertex_normals, mesh.vertex_areas, normal_stiffness, n_modes)

save_figure(render.draw_torus_fields(mesh, [transported_field, modes[0], modes[2]]), "connections_modes");

## 5. The curvature inside a face is a turn

A tangent vector field on a closed surface must vanish or spin somewhere, unless the surface is a torus: a hair on every point of a ball cannot all lie flat, the hairy ball theorem. Each place where a field vanishes has an index, how many times the field turns as one walks once around it, and the indices always add up to a number fixed by the surface's shape, its Euler characteristic: two for a sphere, zero for a torus. The trivial connections of Crane, Desbrun and Schröder design such fields: any choice of places and indices that adds up to that number has a field, and the smoothest one is found by one linear solve.

Carried by parallel transport around the three edges of a triangle and back to the start, a tangent vector comes back turned in its tangent plane, because the surface is curved inside the triangle: the product of the three rotors is a turn about the normal, by the face's holonomy. On a sphere the holonomy of a face is the solid angle its three normals span, and the holonomies of all faces add up to `2 pi` times the Euler characteristic, two whole turns, the discrete theorem of Gauss and Bonnet.

A whole turn of a rotor is minus one, a half turn of its own, so the angle of a loop is read from the square of its rotor, the turn itself, by the plane of its logarithm about the normal: `(loop * loop).log() | normal.dual()`, an angle between minus and plus half a turn.

In [ ]:
def turn(rotors: Rotor, normals: Vector) -> Scalar:
    """The angle by which each rotor turns tangent vectors about its unit normal, read from its square."""
    return (rotors * rotors).log() | normals.dual()                              # [...] Scalar


def loop_turn(normals: Vector) -> Scalar:
    """The holonomy of transport around closed loops of unit normals `[..., steps]`."""
    following = concatenate([normals[..., 1:], normals[..., :1]], axis=-1)      # [..., steps] Vector
    steps = transport_step(normals, following)                                   # [..., steps] Rotor
    # The running product of the steps, each later step multiplying from the left.
    around = steps.cumprod(axis=-1)[..., -1]                                     # [...] Rotor
    return turn(around, normals[..., 0])                                         # [...] Scalar


sphere_normals = sphere.vertex_normals.batch()                                  # [V] Vector
curvature = loop_turn(sphere_normals[sphere.faces])                             # [F] Scalar

render.print_scalars("the faces' holonomies, in whole turns:", curvature.sum(axis=0) / (2 * np.pi))

## 6. An extra turn on every edge

A connection is trivial when carrying a vector around any loop returns it unchanged: then carrying one vector out from anywhere gives a field that does not turn at all along the surface. The surface's own transport is not trivial, because the surface is curved, so each edge gets an extra turn by some angle in the head's tangent plane, `(n_head.dual() * (-angle / 2)).exp()`. A turn in a tangent plane commutes with transport into that plane, so the holonomy of a loop grows by exactly the sum of the extra turns along it, each counted forwards or backwards by the direction in which the loop runs along the edge: the circulation of the edge angles around the loop.

Each face is asked to turn by `2 pi` times its chosen index, its own curvature included, so the circulation of the extra turns around every face must be `2 pi` times its index less its curvature: a sparse linear system of one row per face, coupling each face to its three edges. The smallest edge angles that meet it, by least squares, also turn the field the least. A face with index one turns a vector once as one walks around it: the field flows out from it, into it, or around it.

In [ ]:
def adjusted_transport(mesh: Mesh, angles: Scalar) -> Rotor:
    """The rotors from each end of each edge to its head `[E, 2]`: the surface's own transport from the tail
    followed by the edge's extra turn in the head's tangent plane, and none at the head."""
    normals = mesh.vertex_normals.batch()                                         # [V] Vector
    tails, heads = normals[mesh.edges[:, 0]], normals[mesh.edges[:, 1]]           # [E] Vector each
    extra = (heads.dual() * (-angles.batch() / 2)).exp()                         # [E] Rotor
    return stack([extra * transport_step(tails, heads), mv.rotor().broadcast_to(extra.shape)], axis=-1)


def trivial_connection(mesh: Mesh, indices: np.ndarray, generators: list[np.ndarray]) -> tuple[Scalar, SparseExtensor, Scalar]:
    """The smallest extra turns on the edges that make every face turn by `2 pi` times its index and every
    generator, a closed loop of vertices around a handle, by nothing; with the circulation and its target."""
    normals = mesh.vertex_normals.batch()                                         # [V] Vector
    curvature = loop_turn(normals[mesh.faces])                                    # [F] Scalar
    # Around each generator, the extra turns undo the surface's own holonomy.
    undone = [-loop_turn(normals[loop[None]]) for loop in generators]             # [1] Scalar each
    # The circulation of edge values around each face, then around each generator.
    circulation = loops(mesh, generators)                                         # [F + G, E] Scalar <- Scalar
    target = concatenate([2 * np.pi * as_scalar(indices) - curvature, *undone]).field()   # Scalar[F + G]
    return circulation.lstsq(target), circulation, target

In [ ]:
# For each scene, the icosahedron's corners whose faces carry a singularity, and the index of each; corners 9
# and 10 lie opposite each other.
sphere_corners = [[9, 10], [9], [9, 4, 8, 3]]
sphere_indices = [[1, 1], [2], [1, 1, 1, -1]]

In [ ]:
sphere_singular = [np.array([faces_around(sphere, corner) for corner in corners]) for corners in sphere_corners]
sphere_designs = [trivial_connection(sphere, face_indices(sphere, singular, indices), [])
                  for singular, indices in zip(sphere_singular, sphere_indices)]
residuals = [circulation(angles) - target for angles, circulation, target in sphere_designs]   # Scalar[F] each

render.print_scalars("the squared circulation left unmet, summed over the faces, per scene:",
                     stack([residual.norm_squared().sites.sum() for residual in residuals]))

## 7. The parallel field

The field itself is the null space of the connection Laplacian of the adjusted transport: `connection` of the adjusted rotors, with each edge weighted by its cotangent ratio. With a trivial connection, a field carried out from one vertex agrees with itself around every loop, so the Laplacian has two fields of energy zero, one a quarter turn of the other in every tangent plane, both of unit length everywhere. The stiff weight along the normals in `low_modes` keeps the normal bundle out of the bottom of the spectrum.

In [ ]:
def parallel_field(mesh: Mesh, angles: Scalar) -> tuple[Scalar, Vector]:
    """The energies of the two smoothest fields of the adjusted transport, and the first as a unit field."""
    _, laplacian = connection(mesh, adjusted_transport(mesh, angles), mesh.edge_ratio)   # [V, V] Vector <- Vector
    energies, fields = low_modes(laplacian, mesh.vertex_normals, mesh.vertex_areas, normal_stiffness, 2)
    field = fields.batch()[0]                                                    # [V] Vector
    return energies, (field / field.norm()).field()                             # [2] Scalar, Vector[V]


sphere_energies, sphere_fields = zip(*[parallel_field(sphere, angles) for angles, _, _ in sphere_designs])
sphere_scenes = list(zip(sphere_fields, sphere_singular, sphere_indices))

render.print_scalars("the energies of the two parallel fields, per scene:\n", stack(sphere_energies))
save_figure(render.draw_fields(sphere, sphere_scenes, render.SPHERE_LIMITS, render.SPHERE_NEEDLE), "connections_fields_spheres");

The arrows show the unit field on the side facing the viewer, and the discs mark singular faces, warm for a positive index and cool for a negative one, larger for a larger index. On the left, two faces of index one at opposite sides of the sphere, one of them in view; in the middle, a single face of index two, the whole sphere's turning gathered into one place; on the right, three faces of index one around a face of index minus one, a saddle, where the field turns once the other way as one walks around it.

## 8. Indices must add up to the Euler characteristic

Around every face the circulation of the extra turns adds up to the circulation of each edge counted once forwards and once backwards, nothing, so the face targets must add up to nothing too: the indices times `2 pi` must equal the curvature of the whole surface, which is the Euler characteristic times `2 pi`. That is the theorem of Poincaré and Hopf, and here it is the condition for the linear system to have a solution. Indices that fall short by one leave one whole turn unmet, which the least-squares solution spreads evenly over all the faces.

In the notation of discrete exterior calculus, the edge angles are a 1-form $\theta$, the face constraints read $d\theta = 2\pi k - K$ with $K$ the discrete Gaussian curvature, and the condition for a solution is $\sum k = \chi$.

In [ ]:
angles, circulation, target = trivial_connection(sphere, face_indices(sphere, np.array([0]), [1]), [])
unmet = circulation(angles) - target                                            # Scalar[F]

render.print_scalars("the unmet turn, summed over the faces, in turns:", unmet.sites.sum() / (2 * np.pi))

## 9. The torus

On a torus the indices add up to zero, so a field with no singularities at all exists. Asking every face for zero is not enough to make the connection trivial, though: a loop that runs once around the tube, or once around the hole, encloses no face, and transport around it could still turn vectors. Those two loops are the torus's generators, and two more rows of the linear system ask each of them to turn by nothing, the extra turns undoing the surface's own holonomy around it. On the left, the torus with no singularities; on the right, a pair of singularities of opposite index on the side facing the viewer.

In [ ]:
# The torus of the designs, by its vertices around the hole and around the tube; for each scene the faces,
# by their place around the hole and around the tube, carrying a singularity, and the index of each.
n_ring_major, n_ring_minor = 40, 20
torus_places = [np.zeros((0, 2), dtype=int), np.array([[32, 2], [0, 2]])]
torus_indices = [[], [1, -1]]

In [ ]:
ring = Mesh.torus(major_radius, minor_radius, n_ring_major, n_ring_minor)
# Its two generators: once around the tube, and once around the hole along the outer equator.
generators = [np.arange(n_ring_minor), np.arange(n_ring_major) * n_ring_minor]
torus_singular = [places[:, 0] * n_ring_minor + places[:, 1] for places in torus_places]
torus_designs = [trivial_connection(ring, face_indices(ring, singular, indices), generators)
                 for singular, indices in zip(torus_singular, torus_indices)]
torus_residuals = [circulation(angles) - target for angles, circulation, target in torus_designs]   # Scalar[F + G] each
torus_energies, torus_fields = zip(*[parallel_field(ring, angles) for angles, _, _ in torus_designs])
torus_scenes = list(zip(torus_fields, torus_singular, torus_indices))

render.print_scalars("the energies of the two parallel fields, per scene:\n", stack(torus_energies))
save_figure(render.draw_fields(ring, torus_scenes, render.TORUS_LIMITS, render.TORUS_NEEDLE), "connections_fields_tori");

## 10. Turning the field

The null space holds a field and its quarter turn, so any turn of the field by one angle about every normal is parallel too. Turning the field with two singularities of index one through a whole turn takes the singularity in view from a source, through a vortex, to a sink, and back.

In [ ]:
# The frames of a whole turn.
n_turning_frames = 48

In [ ]:
turn_angles = np.linspace(0, 2 * np.pi, n_turning_frames, endpoint=False)        # [frames]
planes = sphere_normals.dual()                                                   # [V] Bivector
field, singular, indices = sphere_scenes[0]
turned = ((planes * (-turn_angles[:, None] / 2)).exp() >> field.batch()).field()   # [frames] Vector[V]

show_animation(render.animate_turning(sphere, turned, singular, indices), "connections_turning", duration_ms)

## 11. Stripes

A stripe pattern is a field of evenly spaced stripes running across a surface at right angles to a chosen direction field: the stripes' phase advances along the field at a chosen frequency, as in the stripe patterns of Knöppel, Crane, Pinkall and Schröder. Across each edge the phase should advance by the frequency times the field's component along the edge, the edge's target, and the phase is carried by a tangent vector at every vertex, measured as its angle from the direction field. The direction field of a trivial connection is parallel for that connection, so a tangent vector carried by the same transport and turned further by the edge's target in the head's tangent plane has its angle from the field advance by exactly the target. The stripe field is the smoothest tangent field of that connection, the trivial connection's turns with the targets added: one more set of rotors for `connection`, with each face adding its half-cotangents to its edges' weights only where the direction field does not turn inside it, since across a singular face the field's average means nothing.

In [ ]:
# The finer sphere of the stripes, and the stripes' frequency on it.
n_stripe_subdivisions = 4
frequency = 20.0

In [ ]:
def targets(mesh: Mesh, field: Vector, frequency: float) -> Scalar:
    """The phase advance across every edge `Scalar[E]`: the frequency times the field paired with the
    edge's displacement, the field averaged over the edge's two ends."""
    ends = field.batch()[mesh.edges]                                             # [E, 2] Vector
    return ((ends.sum(axis=-1) / 2 & mesh.edge_displacements.batch()) * frequency).field()


def stripes_along(mesh: Mesh, index: np.ndarray) -> tuple[tuple[Vector, Vector, Scalar], Scalar]:
    """One family of stripes along the parallel field of the trivial connection with every face turning
    by its index: the field's line, its unit stripe field, and the edges' targets `[V, 1]`, `[V, 1]`,
    `[E, 1]`; and the energies of the two smoothest stripe fields."""
    angles, _, _ = trivial_connection(mesh, index, [])
    _, field = parallel_field(mesh, angles)                                      # Vector[V]
    advance = targets(mesh, field, frequency)                                    # Scalar[E]
    # Every face is regular except the singular ones.
    regular = as_scalar(np.where(index == 0, 1.0, 0.0)).field()                 # Scalar[F]
    _, laplacian = connection(mesh, adjusted_transport(mesh, angles + advance), mesh.weighted_edge_ratio(regular))
    energies, stripe_fields = low_modes(laplacian, mesh.vertex_normals, mesh.vertex_areas, normal_stiffness, 2)
    stripe = stripe_fields.batch()[0]                                            # [V] Vector
    return (field.batch()[:, None], (stripe / stripe.norm())[:, None], advance.batch()[:, None]), energies


fine = Mesh.icosphere(n_stripe_subdivisions)
fine_singular = [np.array([faces_around(fine, corner) for corner in corners]) for corners in sphere_corners]
stripe_patterns, stripe_energies = zip(*[stripes_along(fine, face_indices(fine, singular, indices))
                                         for singular, indices in zip(fine_singular, sphere_indices)])

views = np.broadcast_to(render.VIEW, (len(stripe_patterns), 2))
save_figure(render.draw_patterns(fine, stripe_patterns, render.draw_bands, render.SPHERE_REACH, views), "connections_stripes_spheres");

The light and dark bands are the stripes, from the stripe field's angle to the direction field, for the three designed fields of the sphere above. The stripe field comes as a pair, a field and its quarter turn, two patterns shifted by a quarter of a stripe. Where the targets around a face add up to more than the face's singularity allows, no phase can follow them all, the lowest eigenfield gives up there, and the stripes fork.

A cross field is a pair of lines at right angles at every vertex, which may turn by a quarter or a half turn around a singularity. A trivial connection whose faces turn by quarter turns gives one: a unit tangent vector carried out from the first vertex along a breadth-first tree, in one sparse solve, lands on every head turned by a whole number of quarter turns from the head's own. Stripes run across both lines, but carried across an edge each of the tail's lines lands on one of the head's lines, pointing along it or against it, and a stripe phase along a line reads backwards along the line reversed, so the stripe field carried onto a line it lands against is mirrored. The two families are one field over the cross's two lines at every vertex, coupled across the edges where the lines trade places, and its covariant derivative has three cells in each row. On the left, singularities at the eight corners of a cube, each turning by a quarter; on the right, at the four corners of a tetrahedron inscribed in it, each turning by a half.

In [ ]:
# The cross fields' singularities: at the cube's eight corners, each turning by a quarter, and at the four
# corners of a tetrahedron inscribed in it, each turning by a half.
cube = [[1, 1, 1], [1, 1, -1], [1, -1, 1], [1, -1, -1], [-1, 1, 1], [-1, 1, -1], [-1, -1, 1], [-1, -1, -1]]
cross_corners = [cube, [[1, 1, 1], [1, -1, -1], [-1, 1, -1], [-1, -1, 1]]]
cross_indices = [0.25, 0.5]

In [ ]:
def quarter_turn(mesh: Mesh, field: Vector) -> Vector:
    """The field turned a quarter turn about every vertex normal `Vector[V]`."""
    return (field.batch() | mesh.vertex_normals.batch().dual()).field()          # Vector[V]


def carried_out(mesh: Mesh, angles: Scalar) -> Vector:
    """A unit tangent vector at every vertex `Vector[V]`, carried out from the first vertex by the transport
    of the given turns along a breadth-first tree: one sparse solve, each tree edge asking its head to be
    its tail carried across it, and the first vertex asking to point across its first edge."""
    children, tree = spanning_tree(mesh)
    count = len(children) + 1
    turns = adjusted_transport(mesh, angles)[tree]                               # [V - 1, 2] Rotor
    along_tree = SparseExtensor.from_indices((turns >> Vector) * np.array([-1, 1]), children[:, None], mesh.edges[tree], (count, count))
    first = SparseExtensor.from_indices(as_scalar([1.0]), np.array([0]), np.array([0]), (count, count)) * Vector
    normal = mesh.vertex_normals.batch()[0]                                      # Vector
    # the first edge met with the normal: a tangent direction across it
    seed = (mesh.edge_displacements.batch()[0] | normal).normalized()          # Vector
    return (along_tree + first).solve((seed * as_scalar(np.arange(count) == 0)).field())


def cross_stripes(mesh: Mesh, angles: Scalar, cross: Vector, regular: Scalar, frequency: float
                  ) -> tuple[Vector, Vector, Scalar]:
    """Two families of stripes across the two lines of a cross field: the lines `[V, 2] Vector`, the stripe
    field of each `[V, 2] Vector`, and the targets across every edge for each line at its head `[E, 2] Scalar`."""
    count = mesh.vertex_areas.batch().shape[0]
    forward = adjusted_transport(mesh, angles)[:, 0]                             # [E] Rotor
    lines = stack([cross.batch(), quarter_turn(mesh, cross).batch()], axis=-1)   # [V, 2] Vector
    tails, heads = mesh.edges[:, 0], mesh.edges[:, 1]
    ahead = lines[heads]                                                         # [E, 2] Vector
    # One where the tail's line b lands along the head's line a, minus one where it lands against it, and
    # zero where it lands across it.
    match = ahead[:, :, None] | (forward[:, None] >> lines[tails])[:, None, :]  # [E, 2, 2] Scalar
    behind = (match * lines[tails][:, None, :]).sum(axis=-1)                     # [E, 2] Vector
    advance = (((ahead + behind) / 2) & mesh.edge_displacements.batch()[:, None]) * frequency   # [E, 2] Scalar
    target_turn = (mesh.vertex_normals.batch()[heads][:, None].dual() * (-advance / 2)).exp()   # [E, 2] Rotor
    # Carried, then mirrored onto the head's line where it lands against it.
    moved = forward[:, None, None] >> Vector                                     # [E, 1, 1] Vector <- Vector
    along = ahead[:, :, None]                                                    # [E, 2, 1] Vector
    landed = match * match * moved + (match - match * match) * (along * (along | moved))   # [E, 2, 2] Vector <- Vector
    # The head's stripe field turned back by the target, against the tail's carried and mirrored.
    cells = stack([target_turn.reverse() >> Vector, -landed[..., 0], -landed[..., 1]], axis=-1)   # [E, 2, 3] Vector <- Vector
    rows = np.arange(len(heads))[:, None, None] * 2 + np.arange(2)[None, :, None]           # [E, 2, 1]
    columns = np.stack(np.broadcast_arrays(heads[:, None] * 2 + np.arange(2), tails[:, None] * 2, tails[:, None] * 2 + 1), axis=-1)
    covariant = SparseExtensor.from_indices(cells, rows, columns, (2 * len(heads), 2 * count))   # [2E, 2V] Vector <- Vector
    weights = mesh.weighted_edge_ratio(regular).batch()                          # [E] Scalar
    vertex = np.arange(2 * count) // 2                                           # each line's vertex
    laplacian = covariant.adjoint()(spdiag(weights[np.arange(2 * len(heads)) // 2].field()) * covariant)   # [2V, 2V]
    _, stripe_fields = low_modes(laplacian, mesh.vertex_normals.batch()[vertex].field(),
                                 mesh.vertex_areas.batch()[vertex].field(), normal_stiffness, 1)
    stripe = stripe_fields.batch()[0].reshape(count, 2)                          # [V, 2] Vector
    return lines, stripe / stripe.norm(), advance


def cross(mesh: Mesh, index: np.ndarray) -> tuple[tuple[Vector, Vector, Scalar], Scalar]:
    """The two families of stripes across the cross field of the trivial connection with every face
    turning by its quarter-turn index, and the connection's extra turns on the edges."""
    angles, _, _ = trivial_connection(mesh, index, [])
    regular = as_scalar(np.where(index == 0, 1.0, 0.0)).field()                 # Scalar[F]
    return cross_stripes(mesh, angles, carried_out(mesh, angles), regular, frequency), angles


cross_patterns, cross_angles = zip(*[cross(fine, face_indices(fine, nearest_faces(fine, mv.vector(corners)), quarters))
                                     for corners, quarters in zip(cross_corners, cross_indices)])

views = np.broadcast_to(render.VIEW, (len(cross_patterns), 2))
save_figure(render.draw_patterns(fine, cross_patterns, render.draw_lines, render.SPHERE_REACH, views), "connections_stripes_crosses");

In [ ]:
# checks
# Every vector along the latitude circles stays tangent and turns by the cap's solid angle.
carried = stack(tuple(transported(rings, starts)))                               # [steps + 1, rings] Vector
np.testing.assert_allclose((carried | rings).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((carried[-1] | starts).to_array(), np.cos(2 * np.pi * (1 - np.cos(colatitudes))), atol=2e-3)
# Neither heat diffusion nor the connection Laplacian's tangent modes leak into the normal direction.
np.testing.assert_allclose((diffused | mesh.vertex_normals).to_array(), 0.0, atol=1e-12)
np.testing.assert_allclose((modes | mesh.vertex_normals).to_array(), 0.0, atol=1e-11)
# Quarter-turn rotation in each vertex's tangent plane commutes with the connection.
values = mode_energies.to_array()
np.testing.assert_allclose(values[0], values[1], rtol=1e-10)
np.testing.assert_allclose(values[2], values[3], rtol=1e-10)
edge_weights = spdiag(mesh.edge_ratio)                                           # [E, E] Scalar
quarter_turned = (modes.batch() | normals.dual()).field()                        # [modes] Vector[V]
mode_energy = (edge_weights * covariant(modes).norm_squared()).sites.sum().to_array()
turned_energy = (edge_weights * covariant(quarter_turned).norm_squared()).sites.sum().to_array()
np.testing.assert_allclose(mode_energy, turned_energy, rtol=1e-10)
# The faces' holonomies add up to two whole turns on the sphere.
np.testing.assert_allclose(curvature.sum(axis=0).to_array(), 4 * np.pi, rtol=1e-12)
# Every design meets its circulation, and its parallel fields have no energy.
np.testing.assert_allclose(np.concatenate([residual.to_array() for residual in residuals + torus_residuals]), 0.0, atol=1e-12)
np.testing.assert_allclose(np.concatenate([energies.to_array() for energies in [*sphere_energies, *torus_energies]]), 0.0, atol=1e-12)
# Indices one short of the Euler characteristic leave one whole turn spread over the faces.
np.testing.assert_allclose(unmet.to_array(), 2 * np.pi / len(sphere.faces), rtol=1e-9)
# Every stripe field comes with its quarter turn, of equal energy, and across most edges its phase, the
# angle from the stripe field to the direction field, advances by the edge's target.
for (lines, stripes, advance), energies in zip(stripe_patterns, stripe_energies):
    np.testing.assert_allclose(energies.to_array()[1], energies.to_array()[0], rtol=1e-6)
    sines = ((stripes ^ lines) | fine.vertex_normals.batch()[:, None].dual()).to_array()   # [V, 1]
    phase = np.arctan2(sines, (stripes | lines).to_array())
    steps = phase[fine.edges[:, 1]] - phase[fine.edges[:, 0]] - advance.to_array()
    assert np.median(np.abs(np.angle(np.exp(1j * steps)))) < 0.25
# Carried across every edge, each of the tail's lines lands along or against one of the head's lines, and
# some edges trade the two lines or reverse them.
for (lines, _, _), angles in zip(cross_patterns, cross_angles):
    forward = adjusted_transport(fine, angles)[:, 0]                             # [E] Rotor
    match = (lines[fine.edges[:, 1]][:, :, None] | (forward[:, None] >> lines[fine.edges[:, 0]])[:, None, :]).to_array()
    np.testing.assert_allclose(np.abs(match) * (1 - np.abs(match)), 0, atol=1e-8)
    assert (match[:, 0, 0] < 0.5).any()